In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

class ManualRNNCell(nn.Module):
    """
    A single RNN cell built from scratch using raw weight matrices.
    
    At each timestep t:
        h_t = tanh(x_t @ W_xh + h_{t-1} @ W_hh + b)
    
    Args:
        input_size:  number of features per timestep (1 for univariate)
        hidden_size: dimension of the hidden state vector
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """
        One timestep forward pass.
        
        Args:
            x_t:    current input,       shape (batch, input_size)
            h_prev: previous hidden state, shape (batch, hidden_size)
        
        Returns:
            h_t: new hidden state, shape (batch, hidden_size)
        """
        h_t = torch.tanh(x_t @self.W_xh + h_prev @ self.W_hh + self.b)
        return h_t

cell = ManualRNNCell(input_size = 1, hidden_size =32)
print("Weight matrices:")
print(f"  W_xh shape: {cell.W_xh.shape}")
print(f"  W_hh shape: {cell.W_hh.shape}")
print(f"  b shape:    {cell.b.shape}")

total_params = sum(p.numel() for p in cell.parameters())
print(f"\nTotal parameters: {total_params}")



Weight matrices:
  W_xh shape: torch.Size([1, 32])
  W_hh shape: torch.Size([32, 32])
  b shape:    torch.Size([32])

Total parameters: 1088


In [2]:
import numpy as np
import torch
import torch.nn as nn

# Load preprocessed arrays saved from notebook 01
X_train = np.load("../data/X_train.npy")
X_val   = np.load("../data/X_val.npy")
X_test  = np.load("../data/X_test.npy")
y_train = np.load("../data/y_train.npy")
y_val   = np.load("../data/y_val.npy")
y_test  = np.load("../data/y_test.npy")

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

X_train: (294284, 144)
y_train: (294284,)


In [3]:
# Take one real sequence from our training data
sample_seq = torch.tensor(X_train[0], dtype=torch.float32)  # shape (144,)
sample_seq = sample_seq.unsqueeze(0).unsqueeze(-1)           # shape (1, 144, 1)

print(f"Sequence shape: {sample_seq.shape}")
print(f"  dim 0 = batch size (1 sequence)")
print(f"  dim 1 = timesteps (144)")
print(f"  dim 2 = input size (1 feature)")

# Initialize hidden state to zeros — the network starts with no memory
h = torch.zeros(1, cell.hidden_size)
print(f"\nInitial hidden state shape: {h.shape}")
print(f"Initial hidden state (first 5 values): {h[0, :5].detach().numpy()}")

# Walk through each timestep manually
hidden_states = []
for t in range(sample_seq.shape[1]):
    x_t = sample_seq[:, t, :]       # shape (1, 1)
    h = cell(x_t, h)                # update hidden state
    hidden_states.append(h.detach().clone())

print(f"\nAfter processing all 144 timesteps:")
print(f"Final hidden state (first 5 values): {h[0, :5].detach().numpy()}")
print(f"Total hidden states collected: {len(hidden_states)}")

Sequence shape: torch.Size([1, 144, 1])
  dim 0 = batch size (1 sequence)
  dim 1 = timesteps (144)
  dim 2 = input size (1 feature)

Initial hidden state shape: torch.Size([1, 32])
Initial hidden state (first 5 values): [0. 0. 0. 0. 0.]

After processing all 144 timesteps:
Final hidden state (first 5 values): [ 0.00539985  0.00437471  0.00304556 -0.00615811  0.00213834]
Total hidden states collected: 144


In [4]:
class ManualRNNModel(nn.Module):
    """
    Full forecasting model built on top of ManualRNNCell.
    
    Architecture:
        Input sequence (batch, seq_len, 1)
            -> ManualRNNCell applied at each timestep
            -> Final hidden state (batch, hidden_size)
            -> Linear layer
            -> Scalar forecast (batch, 1)
    
    Args:
        input_size:  features per timestep (1 for univariate)
        hidden_size: dimension of hidden state
    """

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn_cell = ManualRNNCell(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: input sequences, shape (batch, seq_len, input_size)
        
        Returns:
            forecast: shape (batch, 1)
        """

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        h = torch.zeros(batch_size, self.hidden_size)

        for t in range(seq_len):
            x_t = x[:, t, :]
            h = self.rnn_cell(x_t, h)

        out = self.fc(h)
        return out
    
model = ManualRNNModel(input_size=1, hidden_size=32)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")
print(f"  RNN cell:     {sum(p.numel() for p in model.rnn_cell.parameters())}")
print(f"  Linear layer: {sum(p.numel() for p in model.fc.parameters())}")

# Test forward pass with one batch
sample = torch.tensor(X_train[:4], dtype=torch.float32).unsqueeze(-1)
out = model(sample)
print(f"\nInput shape:  {sample.shape}")
print(f"Output shape: {out.shape}")
print(f"Raw outputs:  {out.detach().numpy().flatten()}")

Total parameters: 1121
  RNN cell:     1088
  Linear layer: 33

Input shape:  torch.Size([4, 144, 1])
Output shape: torch.Size([4, 1])
Raw outputs:  [0.09914126 0.09913649 0.09913044 0.09913094]


In [5]:
from torch.utils.data import DataLoader, TensorDataset

def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_t = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 256
train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=False)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)

def train_model(
        model: nn.Module, 
        train_loader: DataLoader,
        val_loader: DataLoader, 
        n_epochs: int, 
        lr: float
) -> dict:
    """
    Train a forecasting model and track loss per epoch.

    Args:
        model:        the PyTorch model to train
        train_loader: training DataLoader
        val_loader:   validation DataLoader
        n_epochs:     number of full passes through training data
        lr:           learning rate for Adam optimizer

    Returns:
        history: dict with keys 'train_loss' and 'val_loss' (lists)
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(n_epochs):
        #--Training
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        #--Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                preds = model(X_batch).squeeze()
                val_loss += criterion(preds, y_batch).item()
        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1:>3}/{n_epochs} | "
                  f"Train loss: {train_loss:.6f} | "
                  f"Val loss: {val_loss:.6f}")

    return history


print("Training ManualRNNModel...")
model = ManualRNNModel(input_size=1, hidden_size=32)
history = train_model(model, train_loader, val_loader, n_epochs=10, lr=0.001)
print("\nDone.")


Training ManualRNNModel...
Epoch   2/10 | Train loss: 0.000046 | Val loss: 0.000050
Epoch   4/10 | Train loss: 0.000035 | Val loss: 0.000033
Epoch   6/10 | Train loss: 0.000026 | Val loss: 0.000026
Epoch   8/10 | Train loss: 0.000021 | Val loss: 0.000019
Epoch  10/10 | Train loss: 0.000018 | Val loss: 0.000017

Done.


In [6]:
import plotly.graph_objects as go
import joblib

# Load the scaler so we can convert normalized values back to real degrees
scaler = joblib.load("../data/scaler.pkl")

# Run the model on the test set
test_loader = make_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

model.eval()
all_preds  = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch).squeeze()
        all_preds.extend(preds.numpy())
        all_actuals.extend(y_batch.numpy())

# Convert lists to arrays
all_preds   = np.array(all_preds)
all_actuals = np.array(all_actuals)

# Inverse transform — convert from [0,1] back to real celsius values
# scaler expects shape (N, 1), so reshape, transform, flatten
preds_celsius   = scaler.inverse_transform(all_preds.reshape(-1, 1)).flatten()
actuals_celsius = scaler.inverse_transform(all_actuals.reshape(-1, 1)).flatten()

# Plot first 1000 timesteps so the pattern is visible
n_plot = 1000

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=actuals_celsius[:n_plot],
    mode="lines",
    name="Actual",
    line=dict(color="steelblue", width=1)
))
fig.add_trace(go.Scatter(
    y=preds_celsius[:n_plot],
    mode="lines",
    name="Predicted",
    line=dict(color="coral", width=1)
))
fig.update_layout(
    title="Manual RNN — predicted vs actual temperature (first 1000 test steps)",
    xaxis_title="Timestep",
    yaxis_title="Temperature (°C)",
    height=450
)
fig.show()

# Summary metrics
mae  = np.mean(np.abs(preds_celsius - actuals_celsius))
rmse = np.sqrt(np.mean((preds_celsius - actuals_celsius) ** 2))
print(f"MAE:  {mae:.3f} °C")
print(f"RMSE: {rmse:.3f} °C")

MAE:  0.152 °C
RMSE: 0.229 °C


In [7]:
torch.save(model.state_dict(), "../data/manual_rnn_model.pt")
print("Model saved")

Model saved
